# 02 · Work queues, acknowledgements, and redelivery

**Level:** beginner · **Time:** 35–45 minutes · **Prerequisite:** notebook 01.

You will share tasks between two consumers, use manual acknowledgements, and simulate a worker that
loses its connection after receiving a job. A work queue distributes deliveries among consumers on
the **same queue**; it does not broadcast a copy to every consumer.

Run cells in order. We use two logical consumers in one Python process to keep this first demonstration
easy to inspect. Their callbacks run on the same thread; these are not two parallel CPU workers.
In a service, each worker process normally owns its own connection. Later lessons explore concurrency.


In [ ]:
from pathlib import Path
import sys
import json
import time
import pika

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course" / "rabbit.py").is_file()), None)
assert ROOT is not None, "Open Jupyter from the RabbitMQ course folder."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message


## 1. Prepare eight jobs

Each job has an application ID. Delivery tags are channel-local acknowledgement handles, not business
IDs. Predict: how many total successful callbacks should eight jobs produce in this no-failure run?


In [ ]:
queue_name = unique_name("work")
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=queue_name, durable=False)
    for job_id in range(8):
        channel.basic_publish(exchange="", routing_key=queue_name,
                              body=json.dumps({"job_id": job_id}).encode())
    assert channel.queue_declare(queue=queue_name, passive=True).method.message_count == 8
print("Eight jobs waiting in", queue_name)


## 2. Register two push consumers and service events until a deadline

`basic_consume` registers a callback. `process_data_events` gives Pika time to receive and dispatch
deliveries. A deadline prevents this notebook cell from waiting forever.

Each channel uses `prefetch_count=1`, allowing at most one unacknowledged delivery per consumer here.
Prefetch controls push deliveries; it has no effect on `basic_get` polling. Acknowledging after processing
lets another job arrive. Do not assume a particular worker distribution or completion order in production.


In [ ]:
completed = []
with connect() as connection:
    worker_channels = [connection.channel(), connection.channel()]
    consumer_tags = []

    def callback_for(worker_name):
        def process(channel, method, properties, body):
            job = json.loads(body)
            completed.append((worker_name, job["job_id"]))
            print(worker_name, "processed job", job["job_id"])
            channel.basic_ack(method.delivery_tag)
        return process

    for index, channel in enumerate(worker_channels, start=1):
        channel.basic_qos(prefetch_count=1)
        tag = channel.basic_consume(queue=queue_name,
                                   on_message_callback=callback_for(f"worker-{index}"),
                                   auto_ack=False)
        consumer_tags.append(tag)
    try:
        deadline = time.monotonic() + 8
        while len(completed) < 8 and time.monotonic() < deadline:
            connection.process_data_events(time_limit=0.2)
        assert sorted(job_id for _, job_id in completed) == list(range(8)), completed
    finally:
        for channel, tag in zip(worker_channels, consumer_tags):
            if channel.is_open:
                channel.basic_cancel(tag)
print("Total processed:", len(completed))


**Observe:** there are eight completions across the consumers. Compare this with broadcast in notebook 03.
Multiple consumers and retries can change application processing order even when a queue enqueues in order.

## 3. Simulate a consumer connection disappearing before acknowledgement

First receive a new job and intentionally omit its acknowledgement. The context manager closes the
connection, standing in for the broker detecting a failed worker. There is no business side effect here;
in a real system a worker might have already changed a database before it died.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.basic_publish(exchange="", routing_key=queue_name,
                          body=json.dumps({"job_id": "recover-me"}).encode())
    first_delivery, _, body = wait_for_message(channel, queue_name)
    print("Received but NOT acknowledged:", json.loads(body))
    assert not first_delivery.redelivered
# Closing a channel/connection returns its unacknowledged deliveries to the queue.


## 4. Recover the job on another connection

Predict the `redelivered` flag. A new connection gets a new channel-local delivery tag, which may
coincidentally have the same integer value as before. Use the tag from this delivery to acknowledge it.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    method, _, body = wait_for_message(channel, queue_name, timeout=5)
    recovered = json.loads(body)
    print("Recovered:", recovered, "| Redelivered:", method.redelivered)
    assert recovered["job_id"] == "recover-me"
    assert method.redelivered
    channel.basic_ack(method.delivery_tag)


## 5. Exercise: explicitly requeue once

Send a message and reject its first delivery using `basic_nack(..., requeue=True)`. Receive it a second
time and acknowledge it. Why would an unconditional requeue in every callback be dangerous?

The solution performs exactly one requeue. A tight retry loop can consume CPU and network bandwidth;
later lessons introduce delayed retries, retry budgets, and a dead-letter queue.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.basic_publish(exchange="", routing_key=queue_name, body=b"retry once")
    method, _, body = wait_for_message(channel, queue_name)
    channel.basic_nack(delivery_tag=method.delivery_tag, requeue=True)
    second, _, body = wait_for_message(channel, queue_name)
    assert second.redelivered and body == b"retry once"
    channel.basic_ack(second.delivery_tag)
    print("One controlled retry completed.")


## 6. Cleanup and teaching checkpoint

We observed **at-least-once processing risk**: a worker may perform a side effect and then lose its
connection before the acknowledgement arrives. The broker can deliver that message again. A stable
application ID and idempotent processing are how later lessons prevent duplicate business effects.

`auto_ack=True` instead treats delivery as handled without waiting for your work to succeed; a consumer
failure after delivery can lose the work. Manual acknowledgements do not by themselves make publishing durable.


In [ ]:
with connect() as connection:
    connection.channel().queue_delete(queue=queue_name)
print("Cleaned up", queue_name)


**Troubleshooting:** an unknown-delivery-tag error usually means you acknowledged twice or on the wrong
channel. A redelivery timeout can mean another consumer took the message; verify that you used this run's
unique queue. Repeated redelivery means the worker is failing or closing before acknowledging.

Official references: [work queues](https://www.rabbitmq.com/tutorials/tutorial-two-python) and
[acknowledgements and prefetch](https://www.rabbitmq.com/docs/confirms).
